In [16]:
import os
import re

from lxml import etree
from lxml.etree import DocumentInvalid, XMLSyntaxError

from pathlib import Path

In [ ]:
relaxng_schema = Path("../resources/tei-epidoc.rng")
relaxng_doc = etree.parse(relaxng_schema)
relaxng = etree.RelaxNG(file=relaxng_schema)

In [26]:
filename = Path("../pdlrefwk/sec00009/002/sec00009.sec002.xml")
entity_regex = re.compile(r"Entity \'(?P<entity>\w+)\' not defined")
entity_replacements = {
    'mdash': '—',
    'sect': '§'
}

In [42]:
test_doc = """
<p>
    <lemma xml:lang="lat" targOrder="U">judices</lemma>: not <gloss>
    judges</gloss>, but rather <gloss>jurors</gloss>. They were persons
    selected by law to try facts (under the presidency of a <foreign
        xml:lang="lat">praetor</foreign> or <foreign xml:lang="lat">judex
    quaestionis</foreign>), and <lemma xml:lang="lat">varied</lemma> in number from a single one to
    fifty or more. They were originally selected from the Senators, but
    C. Gracchus had transferred the right to sit as <foreign
        xml:lang="lat">judices</foreign> to the <foreign xml:lang="lat">
    equites</foreign> or wealthy middle class). Sulla, whose reforms
    went into operation <date
        value="-80">B.C. 80</date>, had restored this right to the
    Senators, and the present case was the first to occur under the new
    system. It was brought in the <foreign xml:lang="lat">Quaestio inter
    sicarios</foreign> (or court for the trial of murder), under the
    presidency of the praetor M. Fannius. 
</p>
"""

from lxml.builder import ElementMaker

E = ElementMaker(namespace="http://www.tei-c.org/ns/1.0",
                nsmap={None: "http://www.tei-c.org/ns/1.0"})
APP = E.app
LEM = E.lem

NS = "{http://www.tei-c.org/ns/1.0}"
def convert_lemma_to_applemma(filename):
    doc = etree.parse(filename)
    for lemma in doc.iterfind(f".//{NS}lemma"):
        lang = lemma.attrib['{http://www.w3.org/XML/1998/namespace}lang']
        replacement = APP(
            LEM(lemma.text, {'{http://www.w3.org/XML/1998/namespace}lang': lang})
        )
        lemma.getparent().replace(lemma, replacement)
    with open(filename, 'wb') as f:
        f.write(etree.tostring(doc))

In [43]:
# rewrite, e.g., <date value="-81">...</date>
# as <date when="-0081">...</date>
def fix_dates():
    pass

In [53]:
def parse_doc(filename):
    try:
        doc = etree.parse(filename)
    except XMLSyntaxError as e:
        msg = e.msg
        entity_match = entity_regex.search(msg)
        
        if entity_match is not None:
            entity = entity_match.group('entity')
            replacement = entity_replacements[entity]
            print(f"Replacing {entity} with {replacement} in {filename}")
            
            with open(filename, 'r') as f:
                filedata = f.read()
            
            fixed = filedata.replace(f'&{entity};', replacement)

            with open(filename, 'w') as f:
                f.write(fixed)

            return parse_doc(filename)
        else: 
            print(e)

In [1]:
def validate_doc(filename):
    try:
        doc = etree.parse(filename)
        relaxng.assertValid(doc)
    except DocumentInvalid as e:
        print(e)